# Advanced Bash

## Quick notes

### What this covers
- Process substitution, heredocs and herestrings
- Subshell `( )` versus group `{ }` and variable scope
- The safety options: `set -e`, `-u`, `-o pipefail`, `-x`, `-n`, `-f`, `noclobber`, with their surprising edge cases
- `shopt`, `$-`, `eval`, `declare -p`, indirect expansion, `printf -v`, `read -r -d`, `IFS`
- Arithmetic, number bases, bit operations, brace expansion, nested quoting
- `command` / `builtin` / `type -a`, aliases in scripts
- `$LINENO`, `FUNCNAME`, `BASH_SOURCE`, a stack-trace helper, `PS4`, `trap ERR/DEBUG/RETURN`
- `source` vs running a script, `export -f`, `bash -c` with arguments, common idioms, a `main "$@"` dispatcher
- A table of bash 3.2 vs 4 vs 5 features

### Key points
- `<(cmd)` gives a command's output a file name; `<<'EOF'` is a literal block of text; `<<<"text"` feeds one string.
- `( )` copies the shell (changes are lost); `{ }` runs in the same shell (changes stay).
- `set -euo pipefail` catches many mistakes, but `-e` has holes. Know them, and do not trust it blindly.
- `eval` runs text as code. Never feed it data you did not write; prefer `printf -v`, `declare`, arrays.
- Always quote: `"$var"`, `"$@"`, `"${arr[@]}"`.
- This machine has **bash 3.2**. No associative arrays, no `mapfile`, no `${v^^}`, no `{01..10}` padding. Section 30 lists what to guard.

### Gotchas (the part worth revising twice)
- `set -e` is switched off inside `if`/`while` conditions, on the left of `&&`/`||`, and in anything called from them. A function called as `if f` ignores `-e` completely.
- `local x=$(cmd)` hides `cmd`'s failure, because `local` itself succeeds. Declare first, assign next.
- `((i++))` returns status 1 when `i` is 0, which kills a `set -e` script on newer bash (bash 3.2 does not exit on `(( ))`, but `let` does).
- `set -u` on bash 3.2 even rejects an empty array expansion `"${arr[@]}"`.
- `cmd | while read ...` runs the loop in a subshell, so variables set inside vanish. Use `< <(cmd)` or a heredoc.
- A leading zero makes a number octal: `$((08))` is an error.
- Aliases are off in scripts unless you `shopt -s expand_aliases`, and take effect only on the next line.

### When it matters
- Writing scripts that fail loudly instead of silently producing wrong results
- Debugging a script you did not write (`set -x`, `PS4`, `trap ERR`)
- Making a script portable between macOS (bash 3.2) and Linux (bash 4/5)
- Building small command-line tools with sub-commands

### One-line recap example
```bash
diff <(sort a.txt) <(sort b.txt) && echo "same after sorting"
```
Related: `04-Scripting-Craft` siblings, `03-Files-And-System/02-Process-And-Job-Control` for signals and traps, `01-Basics/03-Functions-And-Arguments` for functions.

## 01. Process substitution: `<(cmd)` and `>(cmd)`

- `<(cmd)` runs `cmd` and gives you a **file name** (a `/dev/fd/N` path) that you read like a file. Use it where a tool wants a filename, such as `diff`, `comm`, `paste`.
- `>(cmd)` is the reverse: a file name that you write to; whatever is written becomes `cmd`'s input.
- It is a bash feature, not POSIX `sh`. `sh script.sh` on some systems rejects it.
- It keeps the loop in the **current** shell when written as `while read ...; done < <(cmd)`, which a plain pipe does not.

In [1]:
# setup: a %%bash cell magic that keeps ONE bash session alive between cells
import atexit, os, shutil, subprocess, tempfile, uuid
from IPython.core.magic import register_cell_magic

_WORK = tempfile.mkdtemp(prefix="bashnb_")
atexit.register(shutil.rmtree, _WORK, ignore_errors=True)
_ENV = {k: v for k, v in os.environ.items() if k not in ("CLICOLOR", "CLICOLOR_FORCE", "FORCE_COLOR", "PYDEVD_DISABLE_FILE_VALIDATION")}
_ENV.update(LC_ALL="C", TZ="UTC", HOME=_WORK, PS1="", PS4="+ ")
_session = None

def _start():
    global _session
    _session = subprocess.Popen(["bash", "--noprofile", "--norc"], stdin=subprocess.PIPE, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=_WORK, env=_ENV)

def _clean(text):
    return text.replace(os.path.join(_WORK, "script"), "script").replace(_WORK, "<work>")

def _run_persistent(code, timeout=60):
    global _session
    if _session is None or _session.poll() is not None:
        _start()
    token = uuid.uuid4().hex
    path = os.path.join(_WORK, "script")
    with open(path, "w") as fh:
        fh.write(code + "\n")
    _session.stdin.write(". '%s' </dev/null; printf '\\n%s %%d\\n' $?\n" % (path, token))
    _session.stdin.flush()
    out, status = [], None
    import threading
    result = {}
    def reader():
        while True:
            line = _session.stdout.readline()
            if not line:
                result["status"] = None; return
            if line.startswith(token + " "):
                result["status"] = int(line.split()[1]); return
            out.append(line)
    t = threading.Thread(target=reader, daemon=True); t.start(); t.join(timeout)
    os.remove(path)
    if t.is_alive():
        _session.kill(); _session = None
        return "".join(out) + "[timed out]\n", 124
    if result.get("status") is None:       # the shell itself exited (e.g. the cell ran `exit N`)
        code = _session.wait(); _session = None
        return "".join(out) + "(the bash session ended; the next cell starts a fresh one)\n", code
    text = "".join(out)
    if text.endswith("\n\n"):
        text = text[:-1]          # drop the blank line our status marker adds
    return text, result.get("status") if result.get("status") is not None else 1

def _run_fresh(code, timeout=60):
    path = os.path.join(_WORK, "script")
    with open(path, "w") as fh:
        fh.write(code + "\n")
    done = subprocess.run(["bash", "--noprofile", "--norc", path], stdin=subprocess.DEVNULL, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, cwd=_WORK, env=_ENV, timeout=timeout)
    os.remove(path)
    return done.stdout, done.returncode

@register_cell_magic
def bash(line, cell):
    """Run the cell in bash. Flags: fresh (new shell), expect_fail (show non-zero status instead of raising)."""
    flags = line.split()
    text, status = (_run_fresh if "fresh" in flags else _run_persistent)(cell)
    print(_clean(text), end="" if text.endswith("\n") or not text else "\n")
    if status != 0:
        if "expect_fail" in flags:
            print("[exit status %d]" % status)
        else:
            raise RuntimeError("bash cell exited with status %d" % status)

In [2]:
%%bash
diff <(printf 'a\nb\nc\n') <(printf 'a\nx\nc\n')
echo "diff exit status: $?"

echo "what the name looks like:"
echo <(true) | sed 's/[0-9][0-9]*$/N/'

echo "--- comm needs sorted files; sort each side on the fly"
comm -12 <(printf 'pear\napple\nfig\n' | sort) <(printf 'fig\nbanana\napple\n' | sort)

2c2
< b
---
> x
diff exit status: 1
what the name looks like:
/dev/fd/N
--- comm needs sorted files; sort each side on the fly
apple
fig


In [3]:
%%bash
# pipe into while: loop runs in a subshell, the counter is lost
count=0
printf 'a\nb\nc\n' | while read -r line; do count=$((count + 1)); done
echo "after pipe-while:    count=$count"

# process substitution: loop runs in this shell
count=0
while read -r line; do count=$((count + 1)); done < <(printf 'a\nb\nc\n')
echo "after < <(...) loop: count=$count"

# >( ) : tee a stream into a helper, here counting lines
printf 'x\ny\nz\n' | tee >(wc -l | tr -d ' ' > lines.txt) > /dev/null
sleep 0.3
echo "lines counted by >( ): $(cat lines.txt)"
rm -f lines.txt

after pipe-while:    count=0
after < <(...) loop: count=3
lines counted by >( ): 3


## 02. Here-documents in depth

- `<<WORD` feeds the following lines to a command's stdin until a line that is exactly `WORD`.
- Unquoted `WORD`: `$var`, `$(cmd)`, `` `cmd` `` and backslash escapes are expanded inside. Quoted (`<<'WORD'` or `<<"WORD"` or `<<\WORD`): the text is **literal**.
- `<<-WORD` strips leading **tab** characters (not spaces) from each line and from the end marker, so you can indent the block inside functions.
- The end marker must be alone on its line with nothing before it (except tabs for `<<-`) and nothing after. `cat -et` shows tabs as `^I` and line ends as `$`.
- You can pipe from a heredoc, or give a command both a heredoc and a redirection: `cat <<EOF > file`.

In [4]:
%%bash
name="World"
cat <<EOF
Expanded:  Hello, $name! Today is 1+1=$((1 + 1)); shell says $(echo "from a command")
Escaped:   \$name stays a dollar sign, and a backslash needs two: \\
EOF
echo "----"
cat <<'EOF'
Literal:   Hello, $name! $(not run) \n stays as typed
EOF

Expanded:  Hello, World! Today is 1+1=2; shell says from a command
Escaped:   $name stays a dollar sign, and a backslash needs two: \
----
Literal:   Hello, $name! $(not run) \n stays as typed


In [5]:
%%bash
# <<- strips leading TABS only (the heredoc lines below start with real tab characters)
printf 'cat <<-END\n\tindented with a tab\n\t\ttwo tabs\nEND\n' > tabs.sh
cat -et tabs.sh
echo "--- output:"
bash tabs.sh
rm tabs.sh

# heredoc into a pipe and into a file
cat <<'EOF' | sort > sorted.txt
pear
apple
fig
EOF
cat sorted.txt; rm sorted.txt

# two heredocs on one line (their bodies follow in order)
cat <<'ONE'; cat <<'TWO'
first
ONE
second
TWO

cat <<-END$
^Iindented with a tab$
^I^Itwo tabs$
END$
--- output:
indented with a tab
two tabs
apple
fig
pear
first
second


In [6]:
%%bash
# heredoc to a file with a literal body: the recommended way to generate config files
cat > app.conf <<'EOF'
name = "demo"
path = $HOME/data      # stays literal; nothing expanded
EOF
cat app.conf

# same, but with expansion on purpose
port=8080
cat > app2.conf <<EOF
port = $port
EOF
cat app2.conf
rm app.conf app2.conf

# gotcha: a marker indented with spaces is NOT a marker (bash reads to the end of input; some versions also print a warning)
bash -c 'cat <<EOF
text
  EOF
echo unreachable' 2>&1 | sed 's/^bash: line [0-9]*: //'

name = "demo"
path = $HOME/data      # stays literal; nothing expanded
port = 8080
text
  EOF
echo unreachable


## 03. Here-strings: `<<<`

- `cmd <<< "text"` feeds one string (plus a trailing newline) to stdin. It is a one-line heredoc.
- The string is expanded like double quotes, so variables work. Quote it to keep spaces intact.
- Great with `read`: `read a b <<< "one two"`. Unlike `echo "$x" | read a b`, it does **not** use a subshell, so the variables survive.
- A here-string always adds a newline, which can change `wc -c` counts.

In [7]:
%%bash
read -r first second rest <<< "alpha beta gamma delta"
echo "first=[$first] second=[$second] rest=[$rest]"

# pipe version loses the values (read runs in a subshell)
unset a b
echo "1 2" | read a b
echo "after pipe: a=[${a:-}] b=[${b:-}]"
read a b <<< "1 2"
echo "after <<<:   a=[$a] b=[$b]"

printf 'no newline' | wc -c | tr -d ' '
wc -c <<< "no newline" | tr -d ' '
echo "(second is 1 bigger: the here-string added a newline)"

grep -c b <<< $'a\nb\nbb'

first=[alpha] second=[beta] rest=[gamma delta]
after pipe: a=[] b=[]
after <<<:   a=[1] b=[2]
10
11
(second is 1 bigger: the here-string added a newline)
2


## 04. Subshell `( )` vs group `{ }`

- `( list )` forks a child shell. Variables, `cd`, `set` options, `exit` inside it never affect the parent.
- `{ list; }` runs in the same shell. The space after `{` and the `;` (or newline) before `}` are required.
- Use `( )` to isolate: `( cd dir && make )` leaves your cwd alone.
- Use `{ }` to **group** commands so one redirection covers all of them, or so `a || { echo oops; exit 1; }` runs two commands.
- A pipeline puts each stage in a subshell, the last one too (bash 4.2+ can keep the last stage in the current shell with `shopt -s lastpipe`, when job control is off).

In [8]:
%%bash
v=outer
( v=inner; cd /; echo "inside ( ): v=$v cwd=$(pwd)" )
echo "after ( ):  v=$v  cwd is back: $([ "$PWD" != / ] && echo yes)"

{ v=changed; echo "inside { }: v=$v"; }
echo "after { }:  v=$v"

# one redirection for a whole group
{ echo "line 1"; echo "line 2"; } > grouped.txt
cat grouped.txt; rm grouped.txt

# exit inside ( ) leaves the parent running; exit inside { } would end the whole shell (not shown)
( exit 5 ); echo "subshell exit status seen by parent: $?"

inside ( ): v=inner cwd=/
after ( ):  v=outer  cwd is back: yes
inside { }: v=changed
after { }:  v=changed
line 1
line 2
subshell exit status seen by parent: 5


In [9]:
%%bash
# run both in fresh shells where exit really exits
bash -c '( exit 1 ); echo "still here after ( exit )"'
bash -c '{ exit 1; }; echo "never printed"'; echo "second one exited with $?"

still here after ( exit )
second one exited with 1


## 05. `set -e` (errexit): what it does

- With `set -e`, the shell exits as soon as a simple command fails (non-zero status).
- It is popular, but it is **not** a safety net; section 06 lists where it quietly does not apply.
- Always run `set -e` demos in a `fresh` shell, otherwise they would end the notebook's session.
- Turn it off again with `set +e`. Handle an expected failure with `|| true` or `if`.

In [10]:
%%bash fresh expect_fail
set -e
echo "step 1"
false
echo "step 2: never printed"

step 1
[exit status 1]


In [11]:
%%bash fresh
set -e
echo "step 1"
false || true                  # expected failure, handled
if ! grep -q needle /dev/null; then echo "needle missing, carrying on"; fi
set +e
false                          # -e is off: no exit
echo "status of the failure: $?"

step 1
needle missing, carrying on
status of the failure: 1


## 06. `set -e` surprises

- **Conditions:** in `if cmd`, `while cmd`, `until cmd`, `! cmd`, and on the left of `&&`/`||`, a failure does not exit.
- **Inherited:** that suppression applies to everything called from the condition, including functions and subshells. Inside `if f; then`, a failing command in `f` does not stop `f`.
- **`&&` lists:** `false && echo hi` does not exit, but `true && false` does (the last command of the list is not exempt).
- **Command substitution in `local`:** `local x=$(false)` has the exit status of `local` (0).
- **Arithmetic:** `((expr))` and `let` return status 1 when the result is 0. On newer bash, `((i++))` with i=0 then exits a `set -e` script; on 3.2 `(( ))` did not exit in the test below, but `let` did. Do not rely on either; write `i=$((i + 1))`.
- **Pipelines:** only the last command counts, unless `pipefail` is set.

In [12]:
%%bash fresh
set -e
f() { false; echo "f kept going after false"; }
if f; then echo "f reported success"; fi        # -e is ignored inside f
f || true                                       # same thing here
echo ---
false && echo "not printed"                     # does not exit
echo "survived a failing left side of &&"
echo ---
( false; echo "subshell: kept going" ) || echo "handled"   # -e disabled in the subshell too

f kept going after false
f reported success
f kept going after false
---
survived a failing left side of &&
---
subshell: kept going


In [13]:
%%bash fresh expect_fail
set -e
true && false           # last command of the list: this one DOES exit
echo "never printed"

[exit status 1]


In [14]:
%%bash fresh expect_fail
set -e
g() { local x=$(false); echo "local masked the failure, x=[$x]"; }
g
h() { local x; x=$(false); echo "never printed"; }
h
echo "never reached"

local masked the failure, x=[]
[exit status 1]


In [15]:
%%bash fresh expect_fail
set -e
i=0
((i++))               # status 1 here; bash 3.2 does NOT exit on a failing (( ))
echo "bash 3.2: survived ((i++)) with i=0"
let "k = 0"           # let returns 1 when the value is 0 -> errexit
echo "never printed"

bash 3.2: survived ((i++)) with i=0
[exit status 1]


In [16]:
%%bash fresh
set -e
false | true          # pipeline status is that of the last command
stages=("${PIPESTATUS[@]}")
echo "pipeline hid the failure; per-stage statuses: ${stages[*]}"
set -o pipefail
false | true || echo "with pipefail the same pipeline now fails (status $?)"

pipeline hid the failure; per-stage statuses: 1 0
with pipefail the same pipeline now fails (status 1)


## 07. `set -u` (nounset)

- Using an unset variable is an error instead of an empty string. It catches typos such as `$fil` for `$file`.
- Test or default values safely: `${var:-default}`, `${var-}` (empty if unset), `[ -n "${var:-}" ]`.
- `$@` and `$*` are always safe. `$1` is not: use `${1:-}` or check `$#`.
- On **bash 3.2**, an empty array is treated as unset: `"${arr[@]}"` fails under `set -u`. Bash 4.4+ fixed this. Workaround: `${arr[@]+"${arr[@]}"}`.

In [17]:
%%bash fresh expect_fail
set -u
file="data.txt"
echo "good: $file"
echo "typo: $fil"
echo "never printed"

good: data.txt
script: line 4: fil: unbound variable
[exit status 1]


In [18]:
%%bash fresh
set -u
echo "default for unset:  [${maybe:-fallback}]"
echo "empty if unset:     [${maybe-}]"
echo "\$@ is safe:         [$@]"
echo "arg count check:    $# args"

arr=()
echo "safe empty-array expansion: [${arr[@]+"${arr[@]}"}]"
arr=(a "b c")
echo "same idiom, filled: [${arr[@]+"${arr[@]}"}]"

default for unset:  [fallback]
empty if unset:     []
$@ is safe:         []
arg count check:    0 args
safe empty-array expansion: []
same idiom, filled: [a b c]


In [19]:
%%bash fresh expect_fail
set -u
arr=()
echo "about to expand an empty array on bash 3.2..."
echo "[${arr[@]}]"
echo "never printed"

about to expand an empty array on bash 3.2...
script: line 4: arr[@]: unbound variable
[exit status 1]


## 08. `set -o pipefail`

- Normally a pipeline's status is the **last** command's status. `false | true` succeeds.
- With `pipefail` the status is the last non-zero status of any stage.
- `PIPESTATUS` (a bash array) always has every stage's status, with or without `pipefail`. Read it **immediately**; the next command overwrites it.
- Classic trap: `producer | head -1` makes the producer die of SIGPIPE (status 141), so under pipefail the pipeline "fails" even though everything worked.

In [20]:
%%bash
false | true;               echo "default:  status=$?   stages=${PIPESTATUS[*]}"
bash -c 'set -o pipefail; false | true; echo "pipefail: status=$?"'
bash -c 'set -o pipefail; yes | head -1 > /dev/null; echo "yes | head -1 with pipefail: status=$? (141 = 128 + SIGPIPE 13)"'

default:  status=0   stages=1 0
pipefail: status=1
yes | head -1 with pipefail: status=141 (141 = 128 + SIGPIPE 13)


## 09. Tracing: `set -x`, `PS4`, `set -n`

- `set -x` prints each command (after expansion) to stderr, prefixed with `PS4` (default `+ `). Turn off with `set +x`.
- Customise `PS4` to show file and line: `PS4='+ ${BASH_SOURCE##*/}:${LINENO}: '`. Single quotes so it expands at print time.
- `bash -x script.sh` traces a script without editing it.
- `set -n` (or `bash -n script.sh`) only **parses**: it reports syntax errors and runs nothing. Use it as a cheap check.

In [21]:
%%bash fresh
name="demo"
set -x
echo "hello $name"
x=$((1 + 2))
set +x
echo "tracing is off again"

+ echo 'hello demo'
hello demo
+ x=3
+ set +x
tracing is off again


In [22]:
%%bash fresh
PS4='+ ${BASH_SOURCE##*/}:${LINENO}: '
f() { echo "in f with $1"; }
set -x
f one
f two
set +x

+ script:4: f one
+ script:2: echo 'in f with one'
in f with one
+ script:5: f two
+ script:2: echo 'in f with two'
in f with two
+ script:6: set +x


In [23]:
%%bash expect_fail
printf 'echo start\nif true; then\n  echo inside\n' > broken.sh        # missing fi
bash -n broken.sh
echo "bash -n exit status: $?"
printf 'echo fine\n' > ok.sh
bash -n ok.sh; echo "bash -n on a good file: $? (and it printed nothing, it did not run)"
rm broken.sh ok.sh

broken.sh: line 4: syntax error: unexpected end of file
bash -n exit status: 2
bash -n on a good file: 0 (and it printed nothing, it did not run)


## 10. `set -f` (noglob) and `noclobber`

- `set -f` turns off filename expansion: `*` stays a literal `*`. Useful when you handle patterns as data.
- `set -C` (`set -o noclobber`) makes `>` refuse to overwrite an existing file. `>|` overrides it, and `>>` is always fine.
- noclobber also protects against `sort file > file`-style mistakes.
- Opening a file for creation under noclobber is atomic (`O_EXCL`), so it works as a simple lock-file test.

In [24]:
%%bash fresh
touch a.txt b.txt
echo "glob on:  $(echo *.txt)"
set -f
echo "glob off: $(echo *.txt)"
set +f
echo "glob on again: $(echo *.txt)"
rm a.txt b.txt

glob on:  a.txt b.txt
glob off: *.txt
glob on again: a.txt b.txt


In [25]:
%%bash fresh
set -o noclobber
echo one > f.txt
echo two > f.txt                      # refused
echo "second write status: $?"
echo three >| f.txt                   # forced
echo four >> f.txt                    # append is allowed
cat f.txt
# as a lock test: only one writer gets to create the file
( set -C; echo $$ > lock.txt ) 2>/dev/null && echo "got lock"
( set -C; echo $$ > lock.txt ) 2>/dev/null || echo "lock already taken"
rm -f f.txt lock.txt

script: line 3: f.txt: cannot overwrite existing file
second write status: 1
three
four
got lock
lock already taken


## 11. `$-` and `set -o`: which options are on?

- `$-` is a string of the single-letter flags that are currently on: `e` errexit, `u` nounset, `x` xtrace, `f` noglob, `C` noclobber, `m` job control, `i` interactive, `B` brace expansion, `h` hash.
- `set -o` lists the long names with on/off; `set +o` prints commands that would restore the state.
- Save and restore: `old=$(set +o)` then `eval "$old"` later; this is one of the safe uses of `eval`.

In [26]:
%%bash fresh
has() { case $- in *"$1"*) echo "$1 is on";; *) echo "$1 is off";; esac; }
has e; has u; has x
set -eu
has e; has u
set +e +u
has e
has i

e is off
u is off
x is off
e is on
u is on
e is off
i is off


In [27]:
%%bash fresh
set -o | awk '$1=="errexit"||$1=="nounset"||$1=="pipefail"||$1=="noclobber"||$1=="noglob"{print}'
saved=$(set +o)
set -o noglob -o nounset
set -o | awk '$1=="noglob"||$1=="nounset"{print "now:", $0}'
eval "$saved"
set -o | awk '$1=="noglob"||$1=="nounset"{print "restored:", $0}'

errexit        	off
noclobber      	off
noglob         	off
nounset        	off
pipefail       	off
now: noglob         	on
now: nounset        	on
restored: noglob         	off
restored: nounset        	off


## 12. `shopt`: bash-only options

- `shopt -s name` turns an option on, `shopt -u name` off, `shopt -p name` prints how to restore it, `shopt -q name` tests it silently (exit status).
- Handy ones that exist on 3.2: `nullglob` (unmatched glob becomes nothing), `dotglob` (`*` also matches hidden files), `nocaseglob`, `nocasematch`, `extglob` (patterns like `+(a|b)`, `!(x)`), `expand_aliases`, `failglob`, `xpg_echo`.
- `extglob` must be switched on in an earlier command than the pattern that uses it: a whole function or `if` block is parsed before `shopt` runs inside it.
- Not on 3.2: `globstar`, `lastpipe`, `autocd`, `checkjobs`, `direxpand`... (Section 30 explains how to check for missing ones.)

In [28]:
%%bash
shopt | awk '{print $1}' | tr '\n' ' '; echo
echo "---"
mkdir g && cd g
touch .hidden vis.txt other.log
echo "default *:   $(echo *)"
shopt -s dotglob;   echo "dotglob *:   $(echo *)"; shopt -u dotglob
echo "no match:    $(echo *.nope)"
shopt -s nullglob;  echo "nullglob:    [$(echo *.nope)]"; shopt -u nullglob
shopt -q extglob && echo "extglob on" || echo "extglob is off by default"
cd .. && rm -rf g

cdable_vars cdspell checkhash checkwinsize cmdhist compat31 dotglob execfail expand_aliases extdebug extglob extquote failglob force_fignore gnu_errfmt histappend histreedit histverify hostcomplete huponexit interactive_comments lithist login_shell mailwarn no_empty_cmd_completion nocaseglob nocasematch nullglob progcomp promptvars restricted_shell shift_verbose sourcepath xpg_echo 
---
default *:   other.log vis.txt
dotglob *:   .hidden other.log vis.txt
no match:    *.nope
nullglob:    []
extglob is off by default


In [29]:
%%bash
shopt -s extglob
mkdir g && cd g
touch a.c b.c c.h d.txt
echo "all but .c:      $(echo !(*.c))"
echo ".c or .h:        $(echo *.@(c|h))"
echo "one-or-more a-d: $(echo +([a-d]).c)"
shopt -s nocasematch
case "HELLO" in hello) echo "nocasematch: matched";; esac
shopt -u nocasematch extglob
cd .. && rm -rf g

all but .c:      c.h d.txt
.c or .h:        a.c b.c c.h
one-or-more a-d: a.c b.c
nocasematch: matched


## 13. `eval`: risks and safe uses

- `eval "string"` expands the string, then runs the result as code. It processes data as **commands**.
- If any part comes from a user, a file name or the network, that part can add commands (code injection).
- Safe uses: running something you built yourself, restoring saved shell state (`eval "$(set +o)"`), and using `printf %q` to quote a value before it goes into an `eval`.
- Usually avoidable: use arrays for commands, `printf -v` or `declare` for dynamic variable names, `${!name}` to read them.

In [30]:
%%bash
name='x; echo INJECTED'
echo "unsafe, eval \"echo \$name\":"
eval "echo $name"
echo "safe, quote the value with printf %q:"
eval "echo $(printf '%q' "$name")"
echo "---"
echo "%q output: $(printf '%q' "it's a \"test\" with spaces")"

unsafe, eval "echo $name":
x
INJECTED
safe, quote the value with printf %q:
x; echo INJECTED
---
%q output: it\'s\ a\ \"test\"\ with\ spaces


In [31]:
%%bash
# build a command in an array, no eval needed
cmd=(printf '%s|' "a b" "c d")
"${cmd[@]}"; echo

# dynamic variable names: three ways
n=color
eval "$n=red"        # eval: only safe if $n is trusted
echo "eval:     $color"
printf -v "$n" '%s' blue     # printf -v: no eval, but still check that $n is a plain name (a value like 'a[$(cmd)]' can run code)
echo "printf -v: $color"
echo "indirect read: ${!n}"

a b|c d|
eval:     red
printf -v: blue
indirect read: blue


## 14. Inspecting variables and functions: `declare -p`, `-f`, `-F`

- `declare -p name` prints a variable **with its attributes** in a form you can paste back: type (`-a` array, `-i` integer, `-r` read-only, `-x` exported).
- `declare -f name` prints a function's source; `declare -F` lists function names only. `typeset` is the same command (ksh name).
- `declare -i n` makes assignments arithmetic: `n=3+4` stores 7 (and a later `n=abc` stores 0, so keep integer variables apart from text ones). `declare -r` makes a variable read-only; changing it is an error.
- `declare` inside a function makes the variable **local** (like `local`).

In [32]:
%%bash
declare -i cnt=5; cnt=cnt+3
declare -r ro=1
declare -x ex=2
arr=(a "b c")
declare -p cnt ro ex arr

greet() { echo "hi $1"; }
declare -f greet
echo "functions named greet*: $(declare -F | awk '$3 ~ /^greet/ {print $3}')"
{ ro=2; } 2>&1
echo "type of a variable: $(declare -p arr | cut -c1-12)"

declare -i cnt="8"
declare -r ro="1"
declare -x ex="2"
declare -a arr='([0]="a" [1]="b c")'
greet () 
{ 
    echo "hi $1"
}
functions named greet*: greet
script: line 10: ro: readonly variable
type of a variable: declare -a a


## 15. Indirect expansion `${!name}` and friends

- `${!name}` uses the **value** of `name` as the variable to read. `name=foo; foo=bar; echo ${!name}` prints `bar`.
- `${!prefix*}` (or `@`) lists variable names that start with `prefix`.
- `${!arr[@]}` lists an array's **indices** (the keys). Handy for sparse arrays.
- Bash 3.2 has no namerefs (`declare -n`, 4.3+) and no associative arrays; indirection and prefixed names (`cfg_host`, `cfg_port`) are the 3.2 substitute.

In [33]:
%%bash
name=cfg_host; cfg_host=example.org; cfg_port=8080
echo "value via indirection: ${!name}"
echo "variables starting with cfg_: ${!cfg_*}"

# poor-man's lookup table using prefixed names
for key in host port; do v="cfg_$key"; echo "$key = ${!v}"; done

sparse=(); sparse[2]=two; sparse[7]=seven
echo "indices: ${!sparse[@]}   count: ${#sparse[@]}"

value via indirection: example.org
variables starting with cfg_: cfg_host cfg_port
host = example.org
port = 8080
indices: 2 7   count: 2


## 16. `printf -v` and `read -r -d`

- `printf -v var 'format' args` stores the formatted text in `var` instead of printing it, with no subshell (faster than `var=$(printf ...)`) and no trailing-newline stripping.
- `read -r` stops backslashes being treated as escapes. Almost always use `-r`.
- `read -d DELIM` reads until `DELIM` instead of newline. `read -d ''` reads until a NUL, which in a heredoc means "everything": a way to load a multi-line block into a variable. It returns status 1 at EOF, so guard it under `set -e`.
- `IFS= read -r line` keeps leading and trailing whitespace on the line.

In [34]:
%%bash
printf -v padded '%05d|%-6s|%6.2f' 42 ab 3.14159
echo "[$padded]"

read -r -d '' block <<'EOF'
first line
  second line, indented
EOF
echo "read -d '' status: $? (1 is normal at EOF)"
echo "block has $(printf '%s\n' "$block" | wc -l | tr -d ' ') lines:"
printf '%s\n' "$block"

IFS= read -r keep <<< "   spaces kept   "
read -r trimmed <<< "   spaces trimmed   "
echo "[$keep] vs [$trimmed]"

IFS=: read -r -d ':' first rest <<< "a:b:c"
echo "first=[$first]"

[00042|ab    |  3.14]
read -d '' status: 1 (1 is normal at EOF)
block has 2 lines:
first line
  second line, indented
[   spaces kept   ] vs [spaces trimmed]
first=[a]


## 17. `IFS` tricks

- `IFS` (Internal Field Separator) decides how unquoted expansions and `read` split text. Default: space, tab, newline.
- Set it **for one command only** with `IFS=, read ...`; it is restored automatically. Setting it globally is the source of many bugs.
- `"${arr[*]}"` joins array elements with the **first character** of `IFS`; `"${arr[@]}"` keeps them separate.
- To split a string into an array: `IFS=, read -r -a parts <<< "$str"`.
- A change to `IFS` in a function can leak; scope it with `local IFS=...` or run in a subshell.

In [35]:
%%bash
line="alice:x:1000:Alice Smith:/home/alice"
IFS=: read -r user pw uid gecos home <<< "$line"
echo "user=$user uid=$uid gecos=[$gecos]"

IFS=, read -r -a parts <<< "red,green,blue"
echo "parts: ${#parts[@]} -> ${parts[1]}"

join() { local IFS="$1"; shift; echo "$*"; }
join , a b c
join ' / ' x y z      # only the first character of IFS is used as the separator
join '|' 1 2 3

echo "IFS unchanged afterwards: $(printf '%s' "$IFS" | od -An -c | tr -s ' ')"

user=alice uid=1000 gecos=[Alice Smith]
parts: 3 -> green
a,b,c
x y z
1|2|3
IFS unchanged afterwards:  \t \n 


## 18. Arithmetic: `$(( ))`, `(( ))`, `let`

- `$(( expr ))` gives the value; `(( expr ))` is a command whose status is 0 if the result is non-zero, 1 if it is zero. `let "x = x + 1"` is the older form.
- Integers only, 64-bit. Division truncates toward zero: `$((-7/2))` is `-3`, and `%` takes the sign of the left side.
- Inside `$(( ))` variables need no `$`.
- Operators as in C: `+ - * / % **`, comparisons, `&& || !`, `?:`, assignments `+= ++` and bit operations.
- Floating point does not exist in the shell; use `awk` or `bc`.

In [36]:
%%bash
a=17; b=5
echo "sum=$((a + b)) diff=$((a - b)) prod=$((a * b)) quot=$((a / b)) rem=$((a % b)) pow=$((2 ** 10))"
echo "negative: $((-7 / 2)) and $((-7 % 3))"
echo "ternary: $((a > b ? a : b))"
x=3; (( x += 4 )); echo "after (( x += 4 )): $x"
let "y = x * 2"; echo "let: y=$y"
(( x == 7 )) && echo "x is 7"
echo "float by awk: $(awk 'BEGIN { printf "%.3f", 10 / 3 }')"
i=0; (( i++ )); echo "(( i++ )) at i=0 has status $? (the value was 0)"

sum=22 diff=12 prod=85 quot=3 rem=2 pow=1024
negative: -3 and -1
ternary: 17
after (( x += 4 )): 7
let: y=14
x is 7
float by awk: 3.333
(( i++ )) at i=0 has status 1 (the value was 0)


## 19. Number bases and bit operations

- `base#number` reads a number in another base: `$((16#ff))` is 255, `$((2#1010))` is 10. Bases 2 to 64.
- A leading `0` means **octal** and `0x` means hex: `$((010))` is 8, and `$((08))` is an error.
- Remove the leading zeros with `$((10#$n))` when reading zero-padded input such as `08`, `09`.
- Bit operators: `& | ^ ~ << >>`. `printf '%x'`, `%o`, `%d` convert for display.
- There is no binary output in `printf`; build it with a small loop or `bc`.

In [37]:
%%bash
echo "16#ff=$((16#ff)) 2#1010=$((2#1010)) 8#17=$((8#17)) 0x1F=$((0x1F)) 010=$((010))"
echo "and=$((12 & 10)) or=$((12 | 10)) xor=$((12 ^ 10)) not=$((~12)) shl=$((1 << 4)) shr=$((256 >> 3))"
printf 'dec 255 as hex %x, octal %o, hex upper %X\n' 255 255 255

zp=08
echo "10#$zp = $((10#$zp))"
bash -c 'zp=08; echo $((zp))' 2>&1 | sed 's/^bash: //'

# test a bit flag
flags=5   # 101
(( flags & 4 )) && echo "bit 2 (value 4) is set"
(( flags & 2 )) || echo "bit 1 (value 2) is clear"

# binary string by loop
bin() { local n=$1 s=""; while (( n > 0 )); do s=$((n & 1))$s; n=$((n >> 1)); done; echo "${s:-0}"; }
echo "10 in binary: $(bin 10), 255: $(bin 255), 0: $(bin 0)"

16#ff=255 2#1010=10 8#17=15 0x1F=31 010=8
and=8 or=14 xor=6 not=-13 shl=16 shr=32
dec 255 as hex ff, octal 377, hex upper FF
10#08 = 8
08: value too great for base (error token is "08")
bit 2 (value 4) is set
bit 1 (value 2) is clear
10 in binary: 1010, 255: 11111111, 0: 0


## 20. Brace expansion

- `{a,b,c}` expands to separate words; `{1..5}` and `{a..e}` make sequences. It happens **before** variable expansion, so `{1..$n}` does not work.
- Combine with text: `file{1,2}.txt`, `mkdir -p proj/{src,docs,tests}`.
- **Bash 3.2 does not zero-pad** `{01..05}` and does not support a step `{1..10..3}`. Both arrived in bash 4.0. For padded numbers use `seq -w` (BSD and GNU) or `printf '%02d'`.
- Braces inside quotes are literal.

In [38]:
%%bash
echo {1..5}; echo {a..e}; echo {5..1}; echo pre{x,y,z}post
echo "padding on 3.2:   {01..05} -> $(echo {01..05})"
echo "step on 3.2:      {1..10..3} -> $(echo {1..10..3})"
echo "seq -w works:     $(seq -w 1 5 | tr '\n' ' ')"
echo "printf padding:   $(for i in 1 2 3 4 5; do printf '%02d ' $i; done)"
n=3
echo "variable in range: {1..$n} -> $(echo {1..$n})   (3.2 does not expand it)"
echo "via seq:           $(seq 1 $n | tr '\n' ' ')"
mkdir -p proj/{src,docs,tests}; printf '%s ' proj/*; echo; rm -rf proj
echo "quoted braces: '{a,b}' stays literal"

1 2 3 4 5
a b c d e
5 4 3 2 1
prexpost preypost prezpost
padding on 3.2:   {01..05} -> 1 2 3 4 5
step on 3.2:      {1..10..3} -> {1..10..3}
seq -w works:     1 2 3 4 5 
printf padding:   01 02 03 04 05 
variable in range: {1..3} -> {1..3}   (3.2 does not expand it)
via seq:           1 2 3 
proj/docs proj/src proj/tests 
quoted braces: '{a,b}' stays literal


## 21. Nested quoting patterns

- Single quotes: everything literal, no escapes. You cannot put a `'` inside; close, add `\'`, reopen: `'it'\''s'`.
- Double quotes: `$`, `` ` ``, `\`, `!` (interactive) and `"` keep meaning; everything else is literal.
- `$( )` starts a **new** quoting context, so you can nest double quotes inside: `"$(echo "a b")"`. Backticks make this much harder.
- `$'...'` (ANSI-C quoting) understands `\n`, `\t`, `\'`, `\xHH`. It works on bash 3.2.
- To pass arbitrary text through a second shell (`bash -c`, `ssh`), quote it with `printf %q`.

In [39]:
%%bash
echo 'single: $HOME stays $HOME, and "quotes" are fine'
echo 'it'\''s a quote inside single quotes'
echo "double: \$HOME escaped, \"inner\" quotes, and a backslash \\"
echo "nested: $(echo "inner $(echo "innermost") text")"
echo $'ANSI-C: tab[\t] newline[\n] apostrophe[\']'
printf '%s\n' "multi word" 'and $pattern*'

arg='a b; echo oops'
bash -c "echo $(printf '%q' "$arg")"            # safe
bash -c "echo $arg" 2>&1 | head -3              # unsafe: runs 'echo oops'

single: $HOME stays $HOME, and "quotes" are fine
it's a quote inside single quotes
double: $HOME escaped, "inner" quotes, and a backslash \
nested: inner innermost text
ANSI-C: tab[	] newline[
] apostrophe[']
multi word
and $pattern*
a b; echo oops
a b
oops


## 22. `command`, `builtin`, `type -a`

- `type -a name` lists **everything** that `name` could mean, in lookup order: alias, keyword, function, builtin, file in `$PATH`.
- `command name args` skips functions and aliases and runs the builtin or program. `command -v name` tells you what would run (script-friendly, exit status 1 if nothing).
- `builtin name args` runs only the builtin.
- Use these to wrap a command while still calling the original: a function named `cd` calls `builtin cd`.
- A function with the same name as a program shadows it, which is a common source of "why is `ls` acting oddly?".

In [40]:
%%bash
type -a echo
type -t echo; type -t ls; type -t if; type -t nosuchthing || echo "(type -t prints nothing and fails for unknown names)"

cd() { builtin cd "$@" && echo "wrapper: cd worked, now in $(basename "$PWD" | sed 's/^bashnb_.*/<work>/')"; }
mkdir -p sub
cd sub
cd ..
unset -f cd; rmdir sub

echo() { printf 'FUNCTION echo: %s\n' "$*"; }
echo hello
command echo "command echo skips the function"
builtin echo "builtin echo too"
unset -f echo
command -v ls

echo is a shell builtin
echo is /bin/echo
builtin
file
keyword
(type -t prints nothing and fails for unknown names)
wrapper: cd worked, now in sub
wrapper: cd worked, now in <work>
FUNCTION echo: hello
command echo skips the function
builtin echo too
/bin/ls


## 23. Aliases in scripts: `expand_aliases`

- Aliases are **off** in non-interactive shells. Turn them on with `shopt -s expand_aliases`.
- An alias is replaced when a line is **read**. Defining an alias and using it on the same line, or inside the same `{ }`/`( )`/function body that defines it, does not work.
- Prefer a function: it takes arguments properly, works everywhere and needs no `shopt`.
- `unalias name` removes an alias; `alias` alone lists them.

In [41]:
%%bash fresh expect_fail
alias hi='echo hello from alias'
hi
echo "status: $?"

script: line 2: hi: command not found
status: 127


In [42]:
%%bash fresh
shopt -s expand_aliases
alias hi='echo hello from alias'
hi
alias ll='ls -1'
touch one two
ll one two
rm one two
alias hi
unalias hi
type hi 2>&1 | head -1

hello from alias
one
two
alias hi='echo hello from alias'
script: line 10: type: hi: not found


## 24. `$LINENO`, `FUNCNAME`, `BASH_SOURCE`, `BASH_LINENO`

- `$LINENO`: the current line number in the script or function.
- `FUNCNAME`: array of function names, innermost first (`FUNCNAME[0]` is the current function).
- `BASH_SOURCE`: array of source file names matching `FUNCNAME`. `BASH_LINENO[i]` is the line in `BASH_SOURCE[i+1]` from which `FUNCNAME[i]` was called.
- `${BASH_SOURCE[0]}` is the file being run even when sourced, while `$0` is the shell/script that was started. They are equal only when the script is run directly.
- These give you error messages like `script.sh:42 in my_func()`.

In [43]:
%%bash fresh
inner() {
    echo "in ${FUNCNAME[0]}: line $LINENO; called from ${FUNCNAME[1]} at line ${BASH_LINENO[0]}"
}
outer() { inner; }
outer
echo "script file: $(basename "${BASH_SOURCE[0]}")  \$0 is the same: $([ "${BASH_SOURCE[0]}" = "$0" ] && echo yes)"

in inner: line 2; called from outer at line 4
script file: script  $0 is the same: yes


In [44]:
%%bash fresh
# a stack-trace helper: print the call chain
stack_trace() {
    local i
    echo "stack trace:"
    for (( i = 1; i < ${#FUNCNAME[@]}; i++ )); do
        printf '  at %s() [%s:%s]\n' "${FUNCNAME[$i]}" "$(basename "${BASH_SOURCE[$i]}")" "${BASH_LINENO[$((i - 1))]}"
    done
}
c() { stack_trace; }
b() { c; }
a() { b; }
a

stack trace:
  at c() [script:9]
  at b() [script:10]
  at a() [script:11]
  at main() [script:12]


## 25. `trap ERR`, `DEBUG`, `RETURN`

- `trap '...' ERR` runs when a command fails, in the same situations where `set -e` would exit (so it is also silent in conditions). Combine with `set -E` (errtrace) so functions and subshells inherit it.
- `trap '...' DEBUG` runs before **every** simple command. `$BASH_COMMAND` holds the command about to run. Slow, but good for tracing or profiling.
- `trap '...' RETURN` runs when a function returns or a sourced file finishes.
- These pair well with `stack_trace` from section 24.

In [45]:
%%bash fresh expect_fail
set -E
on_err() { echo "ERR trap: status $? from command: $BASH_COMMAND"; }
trap on_err ERR
f() { ls /definitely/not/there 2>/dev/null; echo "f carries on (no -e here)"; }
true
false
echo "continues, since -e is not set"
f
echo "status of last command: $?"
exit 3

ERR trap: status 1 from command: false
continues, since -e is not set
ERR trap: status 1 from command: ls /definitely/not/there 2> /dev/null
f carries on (no -e here)
status of last command: 0
[exit status 3]


In [46]:
%%bash fresh
trap 'echo "  DEBUG about to run: $BASH_COMMAND"' DEBUG
x=1
echo "value is $x"
trap - DEBUG
echo "debug trap removed"

  DEBUG about to run: x=1
  DEBUG about to run: echo "value is $x"
value is 1
  DEBUG about to run: trap - DEBUG
debug trap removed


In [47]:
%%bash fresh
work() { trap 'echo "  RETURN trap: leaving ${FUNCNAME[0]}"' RETURN; echo "  doing work"; }
work
echo "back in main"

  doing work
  RETURN trap: leaving work
back in main


## 26. `source` vs running a script vs `exec`

- `source file` (or `. file`) runs the file in the **current** shell: its variables, functions and `cd` stay. Use it for libraries and config.
- `bash file` or `./file` starts a **child** process: it cannot change your variables or cwd.
- `exec` replaces the current shell with another program (see `03-Files-And-System/02-Process-And-Job-Control`).
- Arguments after `source file a b` become `$1 $2` inside it while it runs. `return` ends a sourced file early; `exit` would end your shell.
- `source name` without a slash searches `$PATH` first, so write `source ./lib.sh` for a file in the current directory.

In [48]:
%%bash
cat > lib.sh <<'EOF'
LIB_LOADED=yes
lib_hello() { echo "hello from lib, arg1=${1:-none}"; }
[ -n "${SKIP_REST:-}" ] && return 0
echo "lib: loading the rest"
EOF

bash ./lib.sh;           echo "after bash ./lib.sh -> LIB_LOADED=[${LIB_LOADED:-unset}]"
source ./lib.sh;         echo "after source        -> LIB_LOADED=[${LIB_LOADED:-unset}]"
lib_hello
SKIP_REST=1 . ./lib.sh first
lib_hello 2
unset -f lib_hello; unset LIB_LOADED SKIP_REST; rm lib.sh

lib: loading the rest
after bash ./lib.sh -> LIB_LOADED=[unset]
lib: loading the rest
after source        -> LIB_LOADED=[yes]
hello from lib, arg1=none
hello from lib, arg1=2


## 27. `export -f` and `bash -c` with arguments

- `export var` makes a variable visible to child processes. `export -f func` does the same for a **function**; the child must be `bash`.
- `bash -c 'cmd' name arg1 arg2`: the first word after the command string becomes `$0` (the "script name" used in error messages), and the rest become `$1`, `$2`...
- A common bug is `bash -c 'echo $1' a b`: it prints `b`, not `a`, because `a` went into `$0`.
- This is also how `find -exec bash -c '...' _ {} +` and `xargs bash -c '...' _` are written; `_` is a throw-away `$0`.
- Exported functions are handy for `xargs -P` jobs but are a (rare) security risk if untrusted environments reach your script.

In [49]:
%%bash
bash -c 'echo "\$0=$0 \$1=$1 \$2=$2 \$#=$#"' zero one two
bash -c 'echo "only args: \$1=$1"' first second
echo "--- exported function"
shout() { echo "SHOUT: $1"; }
bash -c 'shout hi' 2>&1 | sed 's/^bash: //'
export -f shout
bash -c 'shout hi'
printf '%s\n' a b c | xargs -n 1 bash -c 'shout "$0"'
unset -f shout

$0=zero $1=one $2=two $#=2
only args: $1=second
--- exported function
shout: command not found
SHOUT: hi
SHOUT: a
SHOUT: b
SHOUT: c


## 28. Everyday idioms

- `: "${VAR:=default}"` assigns a default if `VAR` is unset or empty. `:` is a no-op that makes the shell run the expansion. Quote it so the result is not run as a command.
- `${VAR:?message}` stops with an error if unset or empty (a required-variable check). `${VAR:+text}` gives `text` only if `VAR` is set and non-empty.
- `${#arr[@]}` is the number of elements; `${#str}` is the string length; `${arr[@]:1}` skips the first element.
- `: > file` empties a file without a subshell. `cat > file <<'EOF'` writes a literal block (section 02).
- `[ -n "${x:-}" ]` is the `set -u`-safe way to test "is set and non-empty".

In [50]:
%%bash fresh expect_fail
: "${COLOR:=blue}"
echo "COLOR=$COLOR"
COLOR=red; : "${COLOR:=blue}"; echo "kept existing: $COLOR"

echo "alt text only if set: [${COLOR:+custom}] and [${NOPE:+custom}]"
arr=(a b c d); f1=hello
echo "count=${#arr[@]} rest=${arr[@]:1} len('hello')=${#f1}"
f=archive.tar.gz
echo "strip suffix: ${f%.gz}  strip all: ${f%%.*}  strip prefix: ${f#*.}"
echo "---"
echo "required var check:"
echo "REQUIRED=${REQUIRED:?REQUIRED must be set}"
echo "never printed"

COLOR=blue
kept existing: red
alt text only if set: [custom] and []
count=4 rest=b c d len('hello')=5
strip suffix: archive.tar  strip all: archive  strip prefix: tar.gz
---
required var check:
script: line 12: REQUIRED: REQUIRED must be set
[exit status 1]


## 29. A small dispatcher: `main "$@"`

- Put the logic in functions, the entry in `main`, and call `main "$@"` once at the bottom. That keeps variables local and the script testable.
- Guard the call with `[ "${BASH_SOURCE[0]}" = "$0" ]` so `source script.sh` loads the functions without running anything.
- A `case "${1:-help}"` picks the sub-command; unknown ones print usage and exit non-zero (2 by convention).
- Send usage and errors to stderr (`>&2`), results to stdout.
- A file meant to be sourced should not call `set -e` itself: it would change the caller's shell. Put the `set` line inside `main`-guarded code if you need it.

In [51]:
%%bash
cat > tool.sh <<'EOF'
#!/usr/bin/env bash
set -euo pipefail

usage() {
    cat >&2 <<'USAGE'
usage: tool.sh <command> [args]
  greet NAME     print a greeting
  sum N...       add numbers
USAGE
}

cmd_greet() { echo "hello, ${1:?name required}"; }
cmd_sum()   { local t=0 n; for n in "$@"; do t=$((t + n)); done; echo "$t"; }

main() {
    local cmd="${1:-help}"; shift || true
    case "$cmd" in
        greet) cmd_greet "$@" ;;
        sum)   cmd_sum "$@" ;;
        help|-h|--help) usage ;;
        *) echo "unknown command: $cmd" >&2; usage; return 2 ;;
    esac
}

if [ "${BASH_SOURCE[0]}" = "$0" ]; then main "$@"; fi
EOF
chmod +x tool.sh
./tool.sh greet Ada
./tool.sh sum 1 2 3 4
./tool.sh help 2>&1

hello, Ada
10
usage: tool.sh <command> [args]
  greet NAME     print a greeting
  sum N...       add numbers


In [52]:
%%bash expect_fail
./tool.sh bogus 2>&1
echo "exit status: ${PIPESTATUS[0]}"
./tool.sh greet; echo "(missing arg status: $?)"

unknown command: bogus
usage: tool.sh <command> [args]
  greet NAME     print a greeting
  sum N...       add numbers
exit status: 2
./tool.sh: line 12: 1: name required
(missing arg status: 1)


In [53]:
%%bash
# sourcing loads the functions but does not run main.
# Do it in a subshell here: tool.sh starts with "set -euo pipefail", and sourcing would switch those on in THIS shell too.
( source ./tool.sh; echo "loaded; callable now: $(cmd_sum 10 20)"; case $- in *e*) echo "after sourcing, errexit is ON in the sourcing shell";; esac )
case $- in *e*) echo "our own shell: errexit on";; *) echo "our own shell: errexit off (the subshell protected it)";; esac
rm tool.sh

loaded; callable now: 30
after sourcing, errexit is ON in the sourcing shell
our own shell: errexit off (the subshell protected it)


## 30. Checking for bash 4+ features safely

- This machine runs bash 3.2. A feature that does not exist is either a **parse error** (the whole script fails to load) or a run-time error.
- To use a newer feature only when available, test `BASH_VERSINFO[0]` and put the new syntax in `eval '...'` so the older parser never sees it.
- Each probe below runs in its own `bash -c`, so errors cannot affect the notebook.

In [54]:
%%bash
probe() { printf '%-22s ' "$1"; bash -c "$2" 2>&1 | head -1 | sed 's/^bash: //; s/^line [0-9]*: //; s/^-c: line [0-9]*: //'; }
probe 'associative array'  'declare -A m'
probe 'case conversion'    's=abc; echo ${s^^}'
probe 'mapfile'            'mapfile -t a </dev/null'
probe 'pipe stderr |&'     'echo hi |& cat'
probe 'negative index'     'a=(1 2 3); echo ${a[-1]}'
probe 'wait -n'            'wait -n'
probe 'globstar'           'shopt -s globstar'
probe 'fractional read -t' 'read -t 0.1 x </dev/null'
probe 'transform ${v@Q}'   'v=x; echo ${v@Q}'

associative array      declare: -A: invalid option
case conversion        ${s^^}: bad substitution
mapfile                mapfile: command not found
pipe stderr |&         syntax error near unexpected token `&'
negative index         a: bad array subscript
wait -n                wait: -n: invalid option
globstar               shopt: globstar: invalid shell option name
fractional read -t     read: 0.1: invalid timeout specification
transform ${v@Q}       ${v@Q}: bad substitution


In [55]:
%%bash
# version-guarded use of a 4+ feature: eval keeps the new syntax away from the old parser
if (( BASH_VERSINFO[0] >= 4 )); then
    eval 's=hello; echo "upper: ${s^^}"'
else
    echo "skipped: needs bash 4+ (this is bash ${BASH_VERSINFO[0]}.${BASH_VERSINFO[1]})"
fi
# portable alternative that works everywhere
s=hello; echo "portable upper: $(printf '%s' "$s" | tr '[:lower:]' '[:upper:]')"

skipped: needs bash 4+ (this is bash 3.2)
portable upper: HELLO


## 31. Bash 3.2 vs 4 vs 5: what changed

macOS ships 3.2 (an old GPLv2 release). Linux distributions usually ship 5.x. Write for 3.2 if the script must run on a stock Mac.

| Feature | 3.2 (this Mac) | 4.x | 5.x |
|---|---|---|---|
| `[[ ]]`, `=~`, `+=`, `$'..'`, `pipefail`, `PIPESTATUS`, `<<<`, `{1..5}` | yes | yes | yes |
| Associative arrays `declare -A` | no | 4.0 | yes |
| `mapfile` / `readarray` | no | 4.0 | yes |
| `${v^^}` `${v,,}` case conversion | no | 4.0 | yes |
| `\|&` , `&>>`, `;;&`, `;&` in `case` | no | 4.0 | yes |
| `coproc`, `globstar`, `$BASHPID` | no | 4.0 | yes |
| `{01..10}` padding, `{1..10..2}` step | no | 4.0 | yes |
| `read -t 0.5` (fractional) | no | 4.0 | yes |
| `printf '%(%F)T'`, `shopt -s lastpipe`, `declare -g`, `[[ -v var ]]` | no | 4.2 | yes |
| Negative array indexes `${a[-1]}`, `wait -n`, `declare -n` (namerefs) | no | 4.3 | yes |
| `${var@Q}` (and other `@` transforms), `local -`, `mapfile -d` | no | 4.4 | yes |
| Empty array under `set -u` works | no (error) | 4.4 | yes |
| `EPOCHSECONDS`, `EPOCHREALTIME`, `SRANDOM`, `BASH_ARGV0`, `wait -f` | no | no | 5.0 |

Entries probed in section 30 (and a few more, such as `declare -n`, `coproc`, `&>>`, `;;&`, `$BASHPID`, `printf '%(%F)T'`, `lastpipe`, `[[ -v ]]`, which also failed in a separate test) were checked on this machine: `declare -A`, `${v^^}`, `mapfile`, `|&`, `${a[-1]}`, `wait -n`, `globstar`, fractional `read -t`, `${v@Q}`, zero padding and steps in brace expansion, and the empty array under `set -u`. The version numbers for when each feature arrived come from the bash release notes and could not be re-checked here; check the bash NEWS file when an exact version matters.

Portable habits: guard with `BASH_VERSINFO`, avoid the 4+ column, use `tr` / `awk` / `sed` for case conversion, `while read` loops for `mapfile`, and prefixed variable names or `case` lookups for associative arrays. Related: `01-Basics/03-Functions-And-Arguments` and the GNU vs BSD tool notes in `02-Streams-And-Text/02-Text-Processing-Tools`.

## Recap
- Process substitution and heredocs/here-strings keep data flowing without temp files; `< <(cmd)` keeps loop variables.
- `( )` isolates, `{ }` groups. Know which you need before you write a `cd` or an `exit`.
- `set -euo pipefail` is a helper, not a guarantee: conditions, `&&` lists, `local x=$(...)`, `((i++))`, pipelines and old-bash empty arrays all have holes.
- Quote everything; avoid `eval`; use `printf %q`, arrays, `printf -v`.
- On macOS bash 3.2 assume none of the bash 4+ features; probe with `BASH_VERSINFO` and use `eval` to hide new syntax.